<a href="https://colab.research.google.com/github/seo5555/seobyeol/blob/main-quest-01/%EB%B8%8C%EB%9D%BC%EC%A7%88_%EC%A0%84%EC%9E%90%EC%83%81_%EA%B1%B0%EB%9E%98_%EB%8D%B0%EC%9D%B4%ED%84%B0_H4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd

df1 = pd.read_csv('/df1.csv')

In [9]:
import os
font_path = '/NanumGothic.ttf'
print('exists:', os.path.exists(font_path), 'cwd:', os.getcwd())

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

fm._load_fontmanager(try_read_cache=False)

fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
family_name = prop.get_name()
print('detected family:', family_name)
mpl.rcParams['font.family'] = family_name
mpl.rcParams['axes.unicode_minus'] = False

exists: True cwd: /content
detected family: NanumGothic


In [13]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# --- 0) 안전 전처리 ---------------------------------------------------------
# 날짜형 변환 (있을 때만)
for c in ["구매일시","예상배송일","실제배송일","리뷰작성일"]:
    if c in df1.columns:
        df1[c] = pd.to_datetime(df1[c], errors="coerce")

# 숫자형 변환 (있을 때만)
for c in ["총결제금액", "배송비합계", "상품가격합계"]:
    if c in df1.columns:
        df1[c] = pd.to_numeric(df1[c], errors="coerce")

# 배송지연일수 없으면 계산
if "배송지연일수" not in df1.columns and {"실제배송일","예상배송일"}.issubset(df1.columns):
    df1["배송지연일수"] = (df1["실제배송일"] - df1["예상배송일"]).dt.days

# --- 1) 재구매여부 생성 ------------------------------------------------------
# 고객별 주문 횟수 계산 (고객고유ID, 주문ID 사용)
df1["고객_주문횟수"] = df1.groupby("고객고유ID")["주문ID"].transform("nunique")
df1["재구매여부"]   = (df1["고객_주문횟수"] > 1).astype(int)   # 0=신규, 1=재구매

# --- 2) H4-1: 신규 vs 재구매 -------------------------------------------------
new_scores = df1.loc[df1["재구매여부"]==0, "리뷰점수"].dropna()
rep_scores = df1.loc[df1["재구매여부"]==1, "리뷰점수"].dropna()

t_stat, p_val = ttest_ind(new_scores, rep_scores, equal_var=False)

pooled_std = np.sqrt((new_scores.var(ddof=1) + rep_scores.var(ddof=1)) / 2)
mean_diff = rep_scores.mean() - new_scores.mean()
cohens_d  = mean_diff / pooled_std

print("==== H4-1 (재구매 vs 신규) ====")
print(f"신규 평균 = {new_scores.mean():.3f}, 재구매 평균 = {rep_scores.mean():.3f}")
print(f"차이 = {mean_diff:.3f}, t = {t_stat:.2f}, p = {p_val:.3e}")
print(f"Cohen's d = {cohens_d:.3f}  (≈0.2 작음 / 0.5 중간 / 0.8 큼)")

# --- 3) H4-2: 지역(주) 차이 ---------------------------------------------------
print("\n==== H4-2 ANOVA (고객주) ====")
anova_model = ols("리뷰점수 ~ C(고객주)", data=df1).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=2)
print(anova_table)

print("\n==== H4-2 Tukey (유의 쌍만) ====")
tukey = pairwise_tukeyhsd(endog=df1["리뷰점수"], groups=df1["고객주"], alpha=0.05)
tukey_df = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
tukey_sig = tukey_df[tukey_df["reject"] == True].copy()
print(tukey_sig.head(30))   # 필요한 만큼 늘려서 확인

# --- 4) 보정 포함 회귀 (지역 고정효과 + 통제변수) -----------------------------
# 상품카테고리가 없으면 C(상품카테고리) 부분을 수식에서 제거하세요.
formula = "리뷰점수 ~ C(고객주) + C(상품카테고리) + 배송지연일수 + 총결제금액 + 재구매여부"
ols_model = ols(formula, data=df1).fit(cov_type="HC3")
print("\n==== H4-2 회귀 (HC3) ====")
print(ols_model.summary())

# 지역 계수만 요약 표로 추출 (상위 효과 확인용)
coefs = ols_model.params.rename("coef").to_frame().join(ols_model.pvalues.rename("p"))
coefs_region = (
    coefs[coefs.index.str.startswith("C(고객주)")].reset_index()
         .assign(고객주=lambda d: d["index"].str.extract(r"C\(고객주\)\[T\.(.+?)\]"))
         [["고객주","coef","p"]]
         .sort_values("coef", ascending=False)
)
print("\n==== H4-2 회귀 지역계수 (상위 15) ====")
print(coefs_region.head(15))

==== H4-1 (재구매 vs 신규) ====
신규 평균 = 4.236, 재구매 평균 = 4.326
차이 = 0.090, t = -4.78, p = 1.791e-06
Cohen's d = 0.076  (≈0.2 작음 / 0.5 중간 / 0.8 큼)

==== H4-2 ANOVA (고객주) ====
                 sum_sq       df          F        PR(>F)
C(고객주)       407.111845     26.0  10.899118  5.341075e-45
Residual  118553.267898  82521.0        NaN           NaN

==== H4-2 Tukey (유의 쌍만) ====
    group1 group2  meandiff   p-adj   lower   upper  reject
99      BA     DF    0.1580  0.0045  0.0229  0.2932    True
101     BA     GO    0.1439  0.0330  0.0045  0.2833    True
103     BA     MG    0.2028  0.0000  0.1056  0.3000    True
104     BA     MS    0.2250  0.0113  0.0222  0.4278    True
105     BA     MT    0.2237  0.0027  0.0376  0.4097    True
108     BA     PE    0.1839  0.0021  0.0328  0.3349    True
110     BA     PR    0.2536  0.0000  0.1439  0.3632    True
111     BA     RJ    0.1066  0.0137  0.0093  0.2039    True
115     BA     RS    0.2363  0.0000  0.1278  0.3449    True
116     BA     SC    0.1833 